# 🔧 Project 5.1 — CNC G-Code Validator
**DSA for Mechatronics · Week 05 — Stacks & Queues**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
CNC machines accept G-code programs containing nested comments `(like this)`,
sub-program calls `[L100]`, and optional blocks `/N10 G01 X10`.
Before machining starts the controller must **validate** every program:
all brackets must be balanced and properly nested.
A **stack** is the natural data structure — push on open, pop and match on close.

Your dataset is a randomly generated batch of G-code programs.
Some are valid; some contain deliberate errors.


## Step 1 — Generate your G-code program batch

In [ ]:
# ── Personalised parameters ─────────────────────────────────────────
N_PROGRAMS   = random.randint(8, 14)
ERROR_PROB   = round(random.uniform(0.25, 0.50), 2)
MAX_DEPTH    = random.randint(2, 4)
MAX_LINES    = random.randint(6, 12)

OPENERS = {'(': ')', '[': ']', '{': '}'}
CLOSERS = {v: k for k, v in OPENERS.items()}

def gen_program(seed_offset, force_error):
    """Generate a short G-code program with nested brackets."""
    random.seed(_seed + seed_offset)
    lines, stack = [], []
    n_lines = random.randint(3, MAX_LINES)
    for ln in range(n_lines):
        tokens = [f"N{(ln+1)*10:03d}"]
        cmd = random.choice(["G00","G01","G02","M03","M05","T01","S1000"])
        tokens.append(cmd)
        # maybe open a bracket
        if len(stack) < MAX_DEPTH and random.random() < 0.35:
            op = random.choice(list(OPENERS))
            stack.append(op)
            tokens.append(op + random.choice(["comment","sub100","cond"]))
        # maybe close a bracket
        elif stack and random.random() < 0.5:
            op = stack.pop()
            tokens.append(random.choice(["end","ok","done"]) + OPENERS[op])
        lines.append(" ".join(tokens))
    # close remaining
    while stack:
        op = stack.pop()
        lines.append(f"N{(len(lines)+1)*10:03d} " + random.choice(["end","ok"]) + OPENERS[op])
    # inject error
    if force_error and lines:
        err_type = random.randint(0, 2)
        idx = random.randint(0, len(lines)-1)
        if err_type == 0:   # extra closer
            lines[idx] += " )"
        elif err_type == 1: # wrong closer
            lines[idx] = lines[idx].replace(")", "]", 1) if ")" in lines[idx] else lines[idx] + " ]"
        else:               # missing closer (drop last close line)
            for i in range(len(lines)-1, -1, -1):
                if any(c in lines[i] for c in OPENERS.values()):
                    lines.pop(i); break
    return lines

programs = []
for i in range(N_PROGRAMS):
    force_err = random.random() < ERROR_PROB
    prog = gen_program(i * 137, force_err)
    programs.append(prog)

random.seed(_seed)  # reset seed for reproducibility downstream

print(f"G-code batch parameters:")
print(f"  Programs generated : {N_PROGRAMS}")
print(f"  Error probability  : {ERROR_PROB*100:.0f}%")
print(f"  Max nesting depth  : {MAX_DEPTH}")
print(f"  Max lines/program  : {MAX_LINES}")
print()
print("Program listing (first 3):")
for i, prog in enumerate(programs[:3]):
    print(f"  ── Program {i+1} ──")
    for line in prog:
        print(f"    {line}")


## Step 2 — Stack-based bracket validator

In [ ]:
def validate_gcode(lines):
    """
    Validate bracket nesting in a G-code program using a stack.
    Push every opener; on closer, pop and check for match.
    Returns (is_valid, error_description, max_depth_reached).
    """
    stack = []
    max_depth = 0
    MATCH = {')': '(', ']': '[', '}': '{'}
    OPENERS_SET = set('([{')
    CLOSERS_SET = set(')]}')

    for line_num, line in enumerate(lines, 1):
        for char in line:
            if char in OPENERS_SET:
                stack.append((char, line_num))
                max_depth = max(max_depth, len(stack))
            elif char in CLOSERS_SET:
                if not stack:
                    return False, f"Unexpected '{char}' on line {line_num} (nothing to close)", max_depth
                top_char, top_line = stack.pop()
                if top_char != MATCH[char]:
                    return False, (f"Mismatch: '{char}' on line {line_num} "
                                   f"closes '{top_char}' opened on line {top_line}"), max_depth

    if stack:
        unclosed = [(c, ln) for c, ln in stack]
        return False, f"Unclosed brackets: {unclosed}", max_depth

    return True, "OK", max_depth

results = []
for i, prog in enumerate(programs):
    valid, msg, depth = validate_gcode(prog)
    results.append({"prog": i+1, "lines": len(prog), "valid": valid,
                    "msg": msg, "depth": depth})

n_valid   = sum(1 for r in results if r["valid"])
n_invalid = N_PROGRAMS - n_valid

print("Validation results:")
print(f"  {'Prog':>5}  {'Lines':>6}  {'Depth':>6}  {'Status':<10}  Detail")
print(f"  {'─'*5}  {'─'*6}  {'─'*6}  {'─'*10}  {'─'*30}")
for r in results:
    status = "✅ VALID" if r["valid"] else "❌ ERROR"
    detail = r["msg"] if not r["valid"] else ""
    print(f"  {r['prog']:5d}  {r['lines']:6d}  {r['depth']:6d}  {status:<10}  {detail}")
print(f"\n  Valid: {n_valid}   Invalid: {n_invalid}")


## Step 3 — RPN expression evaluator for coordinate arithmetic

In [ ]:
def eval_rpn(tokens):
    """
    Evaluate a Reverse Polish Notation expression using a stack.
    Operators: + - * /
    Returns numeric result or raises ValueError on malformed input.
    """
    stack = []
    for tok in tokens:
        if tok in ('+', '-', '*', '/'):
            if len(stack) < 2:
                raise ValueError(f"Not enough operands for '{tok}'")
            b, a = stack.pop(), stack.pop()
            if tok == '+': stack.append(a + b)
            elif tok == '-': stack.append(a - b)
            elif tok == '*': stack.append(a * b)
            elif tok == '/':
                if b == 0: raise ValueError("Division by zero")
                stack.append(round(a / b, 4))
        else:
            stack.append(float(tok))
    if len(stack) != 1:
        raise ValueError("Malformed expression")
    return stack[0]

# Generate personalised RPN coordinate expressions
# (CNC controllers sometimes use RPN for parametric programming)
RPN_EXPRS = []
for i in range(6):
    random.seed(_seed + 500 + i)
    a = random.randint(10, 200)
    b = random.randint(1, 50)
    c = random.randint(2, 10)
    op1 = random.choice(['+', '-'])
    op2 = random.choice(['*', '/'])
    expr = [str(a), str(b), op1, str(c), op2]
    try:
        val = eval_rpn(expr)
        RPN_EXPRS.append((" ".join(expr), val))
    except:
        pass

random.seed(_seed)

print("RPN coordinate expressions (parametric G-code):")
print(f"  {'Expression':<30}  {'Result':>10}")
print(f"  {'─'*30}  {'─'*10}")
for expr_str, val in RPN_EXPRS:
    print(f"  {expr_str:<30}  {val:>10.4f}")


## Step 4 — Nesting depth profile across the batch

In [ ]:
depths       = [r["depth"] for r in results]
max_depth_all = max(depths)
avg_depth     = round(sum(depths) / len(depths), 2)
deepest_prog  = results[depths.index(max_depth_all)]["prog"]

print("Nesting depth profile:")
print(f"  Max depth in batch : {max_depth_all}  (Program {deepest_prog})")
print(f"  Average depth      : {avg_depth}")
print()
# ASCII bar chart
print("  Depth distribution:")
from collections import Counter
depth_counts = Counter(depths)
for d in sorted(depth_counts):
    bar = "█" * depth_counts[d]
    print(f"  depth {d}: {bar} ({depth_counts[d]})")


## 📋 Final Report

In [ ]:
W = 56
print("╔" + "═"*W + "╗")
print("║" + " CNC G-CODE VALIDATION REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<26}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<26}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  BATCH PARAMETERS" + " "*(W-18) + "║")
print(f"║  {'Programs in batch':<26}: {N_PROGRAMS:<26}║")
print(f"║  {'Error probability':<26}: {ERROR_PROB*100:.0f}%{'':<24}║")
print(f"║  {'Max nesting depth':<26}: {MAX_DEPTH:<26}║")
print("╠" + "═"*W + "╣")
print("║  VALIDATION RESULTS" + " "*(W-20) + "║")
print(f"║  {'Valid programs':<26}: {n_valid:<26}║")
print(f"║  {'Invalid programs':<26}: {n_invalid:<26}║")
print(f"║  {'Pass rate':<26}: {n_valid/N_PROGRAMS*100:.1f}%{'':<23}║")
print(f"║  {'Max nesting depth seen':<26}: {max_depth_all}  (Prog {deepest_prog}){'':<15}║")
print(f"║  {'Average nesting depth':<26}: {avg_depth:<26}║")
print("╠" + "═"*W + "╣")
print("║  RPN EVALUATOR" + " "*(W-15) + "║")
print(f"║  {'Expressions evaluated':<26}: {len(RPN_EXPRS):<26}║")
if RPN_EXPRS:
    print(f"║  {'First result':<26}: {RPN_EXPRS[0][1]:<26}║")
    print(f"║  {'Last result':<26}: {RPN_EXPRS[-1][1]:<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which stack or queue concept did you find most important, and why?**
Pick one technique from the notebook and explain in your own words what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
